In [1]:
import numpy as np
import astra
import tomopy

#%matplotlib widget
%matplotlib inline

# --- libraries ---
from pathlib import Path
import os
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import shift
from scipy.ndimage import rotate
import os
import skimage.io as io
import sys
import recast_x as rc
import recast_v2 as nc
import BAMline_recon_v0p2 as bam
import multiprocessing as mp
from multiprocessing import Pool
import psutil
nProc = psutil.cpu_count()
print("number of used cores: ",nProc)
from functools import partial
import time
import scipy as sc

from tqdm.notebook import tqdm_notebook as tqdm

from scipy.ndimage import gaussian_filter, median_filter

import algotom.io.loadersaver as losa
import algotom.prep.correction as corr
import algotom.prep.calculation as calc
import algotom.rec.reconstruction as rec
import algotom.prep.removal as remo
import algotom.prep.filtering as filt
import algotom.util.utility as util

number of used cores:  48


In [2]:
IMAGE_EXTENSIONS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp", ".fits"}

def find_neutron_folders(root_dir, keyword="Neutrons", recursive=True):
    """Return a sorted list of subfolders whose name contains `keyword` (case-insensitive)."""
    root = Path(root_dir)
    it = root.rglob("*") if recursive else root.iterdir()
    folders = [p for p in it if p.is_dir() and keyword.lower() in p.name.lower()]
    return sorted(folders)

def load_image_stack(folder, extensions=IMAGE_EXTENSIONS):
    """Load all images in `folder` (sorted by filename) into a 3D NumPy array
    with shape (n_images, height, width)."""
    folder = Path(str(folder)+"/Proj")
    files = sorted(p for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in extensions)
    if not files:
        raise FileNotFoundError(f"No image files found in {folder}")

    stack = np.stack([io.imread(f) for f in files])
    print(f"Loaded {len(files)} images from {folder} -> stack shape {stack.shape}, dtype {stack.dtype}")
    return stack

def float_to_int16(data, min_gray_value,max_gray_value):
    dif = max_gray_value - min_gray_value
    print("dif: ", dif)
    # ---
    if min_gray_value < 0:
        data = data + np.abs(min_gray_value)
        print("min_abs: ", np.abs(min_gray_value))
        max_gray_value = max_gray_value + np.abs(min_gray_value)
        min_gray_value = 0
    data[data>=max_gray_value]=max_gray_value
    data[data<=min_gray_value]=min_gray_value
    print("max: ", np.max(data))
    print("min: ", np.min(data))
    print("min_gray_value: ", min_gray_value)
    print("max_gray_value: ", max_gray_value)
    # ---
    data_int16 = (((data - min_gray_value) / dif)*(2**16-1)).astype(np.uint16)
    print("max: ", np.max(data_int16))
    print("min: ", np.min(data_int16))
    return data_int16

# --- normalise projections WITH MEDIAN - LOOP ---
def normalise_median(proj_raw, flat_raw, dark_raw):
    # uint to float
    proj_raw = proj_raw.astype(np.float32)
    flat_raw = flat_raw.astype(np.float32)
    dark_raw = dark_raw.astype(np.float32)
    # empty arrays
    normalised_array = np.zeros_like(proj_raw)
    print("normalised_array: ", normalised_array.dtype)
    # generate flat from median
    flat_med = np.median(flat_raw, axis=0)
    #flat_med = sc.signal.medfilt2d(flat_med, kernel_size=3)
    # generate dark from median
    dark_med = np.median(dark_raw, axis=0)
    #dark_med = sc.signal.medfilt2d(dark_med, kernel_size=3)
    for i in range(proj_raw.shape[0]):
        #normalised_array[i,:,:] = (proj_raw[i,:,:]-100)/(flat_med-100)
        normalised_array[i,:,:] = (proj_raw[i,:,:]-dark_med)/(flat_med-dark_med)
    return normalised_array


# --- FBP recon 2D on GPU ---
def FBP_ASTRA_GPU(proj, angle, center, i=None, GPU_numb=None, filter=None):
    # --- None ---
    if filter == None: filter = 'shepp-logan'
    #print('Algorithm: FBP ASTRA')
    #print("slice: ", i)
    rec = np.zeros([proj.shape[1],proj.shape[1]], dtype=np.float32)
    cor_shift = (proj.shape[1] / 2 - center)
    #print("cor_shift: ", cor_shift)
    #print("rec.shape: ",rec.shape)
    # ----- create geometries
    vol_geom = astra.create_vol_geom(proj.shape[1],proj.shape[1])
    proj_geom = astra.create_proj_geom('parallel', 1.0, proj.shape[1], angle)
    # ----- projection geometry with shifted CoR
    proj_geom_cor = astra.geom_postalignment(proj_geom, cor_shift)
    proj_id_cor = astra.create_projector('cuda',proj_geom_cor, vol_geom)
    sinogram_id = astra.data2d.link('-sino', proj_geom_cor, np.ascontiguousarray(proj))
    #sinogram_id, sinogram = astra.create_sino(np.ascontiguousarray(proj), proj_id_cor)    # not sure why not works!
    # change the proj_geom metadata ...
    astra.data2d.change_geometry(sinogram_id, proj_geom_cor)
    # reconstruction
    recon_id = astra.data2d.create('-vol', vol_geom)
    # ----- create configuration 
    cfg = astra.astra_dict('FBP_CUDA')
    cfg['ReconstructionDataId'] = recon_id
    cfg['ProjectionDataId'] = sinogram_id
    cfg['option']={}
    cfg['FilterType'] = filter #'gaussian' #'shepp-logan'
    if GPU_numb is not None:
        GPU_use = i % GPU_numb
        #print("GPU_use: ", GPU_use)
        cfg['option']['GPUindex'] = GPU_use
    else: cfg['option']['GPUindex'] = 0       #decide for 0 (GPU1) or 1 (GPU2)
    cfg['option']['MinConstraint'] = 0
    # ----- Create and run the algorithm object from the configuration structure
    fbp_id = astra.algorithm.create(cfg)
    astra.algorithm.run(fbp_id)
    # ----- get reconstruction result
    rec = astra.data2d.get(recon_id)
    # ----- garbage disposal -----
    astra.data2d.delete(sinogram_id)
    astra.data2d.delete(recon_id)
    astra.projector.delete(proj_id_cor)
    astra.algorithm.delete(fbp_id)
    if i is not None:
        return i, rec
    return rec


# --- 3D reconstruction ---
def recon_3D_FBP(sino, angle, center, filter=None):
    start = time.time()
    recon = np.zeros((sino.shape[1],sino.shape[2],sino.shape[2]))
    #for j in range(recon.shape[0]):
    for j in tqdm(range(recon.shape[0]), desc='reconstruct CT', position=0):
        #if j%100==0: print("recon slice: ", j)
        recon[j,:,:] = FBP_ASTRA_GPU(sino[:,j,:], angle=angle, center=center, i=None, GPU_numb=None, filter=filter)
    end = time.time()
    print("reconstruction time: ", end-start)
    return recon

# --- SIRT recon 2D on GPU ---
def SIRT_ASTRA_GPU(proj, angle, center, i=None, GPU_numb=None, niter=None, minConst=None):
    # --- None ---
    if niter == None: niter = 200
    # --- preparation ---
    rec = np.zeros([proj.shape[1],proj.shape[1]], dtype=np.float32)
    cor_shift = (proj.shape[1] / 2 - center)
    #print("cor_shift: ", cor_shift)
    #print("rec.shape: ",rec.shape)
    # ----- create geometries
    vol_geom = astra.create_vol_geom(proj.shape[1],proj.shape[1])
    proj_geom = astra.create_proj_geom('parallel', 1.0, proj.shape[1], angle)
    # ----- projection geometry with shifted CoR
    proj_geom_cor = astra.geom_postalignment(proj_geom, cor_shift)
    proj_id_cor = astra.create_projector('cuda',proj_geom_cor, vol_geom)
    sinogram_id = astra.data2d.link('-sino', proj_geom_cor, np.ascontiguousarray(proj))
    #sinogram_id, sinogram = astra.create_sino(np.ascontiguousarray(proj), proj_id_cor)    # not sure why not works!
    # change the proj_geom metadata ...
    astra.data2d.change_geometry(sinogram_id, proj_geom_cor)
    # reconstruction
    recon_id = astra.data2d.create('-vol', vol_geom)
    # ----- create configuration 
    cfg = astra.astra_dict('SIRT_CUDA')
    cfg['ReconstructionDataId'] = recon_id
    cfg['ProjectionDataId'] = sinogram_id
    cfg['option']={}
    if GPU_numb is not None:
        GPU_use = i % GPU_numb
        #print("GPU_use: ", GPU_use)
        cfg['option']['GPUindex'] = GPU_use
    else: cfg['option']['GPUindex'] = 0       #decide for 0 (GPU1) or 1 (GPU2)
    if minConst == None:
        cfg['option']['MinConstraint'] = 0
    if minConst is not None:
        print("No Minlog")
    # ----- Create and run the algorithm object from the configuration structure
    sirt_id = astra.algorithm.create(cfg)
    astra.algorithm.run(sirt_id,niter)
    # ----- get reconstruction result
    rec = astra.data2d.get(recon_id)
    # ----- garbage disposal -----
    astra.data2d.delete(sinogram_id)
    astra.data2d.delete(recon_id)
    astra.projector.delete(proj_id_cor)
    astra.algorithm.delete(sirt_id)
    if i is not None:
        return i, rec
    return rec


def load_h5(path_h5,name):
    #f_before, f_after = 20, 20
    with h5py.File(path_h5, 'r') as f:
        prefix = str(name) + "_0001_"
        angles = np.array(f['/4.1/measurement/srot_eh2'])#[20:-20]
        sino = np.array(f['/4.1/measurement/edgezoom'])
        dark = np.array(f['/2.1/measurement/edgezoom'])
        flat = np.array(f['/3.1/measurement/edgezoom'])
        pixel_size = f['/1.1/technique/detector/edgezoom/pixel_size'][0]
        #print("energy: ", round(energy,2))
    return sino, flat, dark, angles, pixel_size

# --- find stitching parameters ---

def mse(img0, img1):
    err = np.sum((img0.astype(float) - img1.astype(float))**2)
    err /= float(img0.shape[0] * img1.shape[1])
    # smaller error means better similarity
    return err

def stich_proj(proj_000, proj_180, overlap=None, moves=None, sub_pixel_resolution=None):
    # -- overlap --
    if overlap == None: overlap = 250
    if moves == None: moves = 1000#500
    if sub_pixel_resolution == None: sub_pixel_resolution = 1
    # -- def reference --
    proj_ref = proj_000[:,-250:-1]
    similarity_array = np.zeros(moves)
    # -- calculate overlap --
    for iter_step in range(moves):
        if iter_step%20 == 0:
            print("iteration step: ", iter_step)
        # -- flip and crop proj_180 --
        proj_flip = np.flip(proj_180[:,-overlap-iter_step:-1-iter_step],axis=1)
        # -- compare --
        '''if iter_step%20 == 0:
            print("proj_ref.shape: ", proj_ref.shape)
            plt.imshow(proj_ref,vmin=7940,vmax=9380)
            plt.show()
            print("proj_flip.shape: ", proj_flip.shape)
            plt.imshow(proj_flip,vmin=7940,vmax=9380)
            plt.show()'''
        similarity_array[iter_step] = mse(img0=proj_ref, img1=proj_flip)
    # -- plot result --
    #plt.plot(similarity_array)
    #plt.show()
    # -- --
    print("proj_ref.shape: ", proj_ref.shape)
    #plt.imshow(proj_ref)#,vmin=7940,vmax=9380)
    #plt.show()
    #
    iter_step = np.argmin(similarity_array)
    print("shift is: ", iter_step)
    proj_flip = np.flip(proj_180[:,-overlap-iter_step:-1-iter_step],axis=1)
    print("proj_flip.shape: ", proj_flip.shape)
    #plt.imshow(proj_flip)#,vmin=7940,vmax=9380)
    #plt.show()
    # -------------------
    # -- stitch result --
    #plt.imshow(np.flip(proj_180[:,:-1-iter_step],axis=1))#,vmin=7940,vmax=9380)
    #plt.show()
    proj_flip = np.flip(proj_180[:,:-1-iter_step-overlap],axis=1)
    proj_stitched = np.concatenate((proj_000, proj_flip),axis=1)
    #plt.imshow(proj_stitched)#,vmin=7940,vmax=9380)
    #plt.show()
    return iter_step, proj_stitched


# --- stiching of all projections - transition slidely ---

def stitch_360_transition(sino_norm, overlap_pixel, crop_offset):
    # --- prepare variables ---
    overlap_pixel_offset = overlap_pixel - 2*crop_offset
    print("overlap_pixel_offset: ", overlap_pixel_offset)
    proj_half = int(sino_norm.shape[0]/2) #2500
    print("proj_half: ", proj_half)
    # ---  ---
    # --- create empty stitching array ---
    array_stitching = np.zeros([int((sino_norm.shape[0]-1)/2),sino_norm.shape[1],2*(sino_norm.shape[2]-overlap_pixel)+overlap_pixel])
    print("array_stitching: ", array_stitching.shape)
    
    # --- make meshgrid ---
    nx, ny = (overlap_pixel_offset, array_stitching.shape[1])
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    xv, yv = np.meshgrid(x, y)
    print("xv: ", xv.shape)
    
    if crop_offset != 0:
        print("hallo")
        # -- pad --
        xv = np.pad(xv, ((0,0),(crop_offset, crop_offset)), 'constant', constant_values=(0, 1))
    print("xv: ", xv.shape)
    #print("xv: ", xv[3,:])
    
    xv_flip = np.flip(xv, axis=1)
    print("xv_flip: ", xv_flip.shape)
    #print("xv_flip: ", xv_flip[5,:])
    # ---------------------------
    # --- loop ---
    for j in range(proj_half):
        #print("j:", j)
        # --- calculate overlap_array ---
        overlap_array = sino_norm[j,:,-overlap_pixel:]*xv_flip  +   np.flip(sino_norm[j+proj_half,:,-overlap_pixel:],axis=1)*xv
        #print("overlap_array: ", overlap_array.shape)
        
        #plt.imshow(overlap_array)
        #plt.gray()
        #plt.show()
        
        # --- concatunate ---
        array_stitching[j,:,:] = np.concatenate((sino_norm[j,:,:-overlap_pixel],overlap_array,np.flip(sino_norm[j+proj_half,:,:],axis=1)[:,overlap_pixel:]),axis=1)
        
        #plt.imshow(array_stitching[j,:,:])
        #plt.gray()
        #plt.show()
    #print("stitching done")
    return array_stitching

# --- ring filtering ---
# function
# remove ring artefacts
def Ring_removal(sino,sigma_=1,i=None):
    # import used libaries
    import sarepy.prep.stripe_removal_original as srm1
    import sarepy.prep.stripe_removal_improved as srm2
    import sarepy.prep.stripe_removal_former as srm3
    # ring removal
    #sino_ring = srm1.remove_stripe_based_sorting(sino, size=70, dim=1)
    sino_ring = srm1.remove_stripe_based_filtering(sino, sigma=sigma_, size=20, dim=1)
    #sino_ring = srm1.remove_stripe_based_fitting(sino, order=5, sigmax=5, sigmay=5)
    if i is not None:
        return sino_ring, i
    return sino_ring

def ring_filter_loop(proj):
    # -- --
    proj_ring = np.zeros_like(proj)
    #for i in range(proj.shape[1]):
    for n in tqdm(range(proj.shape[1]), desc='ring filtering', position=0):
        proj_ring[:,n,:] = Ring_removal(proj[:,n,:])
    return proj_ring

# ----- multiprocessing -----
def Ring_removal_multicore(sino, sigma_, core_numb=None):
    if core_numb is None:
        import psutil
        nProc = psutil.cpu_count()-2
        print("number of used cores: ",nProc)
    else: nProc = core_numb
    pool = Pool(processes=nProc)  # nProc number of used cores
    procs = [pool.apply_async(Ring_removal, args=(sino[:,i1,:], sigma_, i1)) \
            for i1 in range(sino.shape[1])]  # for-loop for 2d array
    #m = np.asarray([p.get() for p in procs])  # store multiprocessing results until end of calculation
    m = [p.get() for p in procs]  # store multiprocessing results until end of calculation
    pool.close()
    pool.join()
    # sort projections:
    result = np.zeros_like(sino)
    for i2 in range(sino.shape[1]):
        result[:,int(m[i2][1]),:] = m[i2][0]
    print("ring filtering done!")
    return result

# ----- def beam harddening -----


def beam_hardening(slide_rec_3D, crop=None,ref_slide=None):
    # --- None ---
    if ref_slide == None: ref_slide = int(slide_rec_3D.shape[0]/2)
    # ---  ---
    slide_rec = np.mean(slide_rec_3D[ref_slide-20:ref_slide+20,:,:],axis=0)
    print("slide_rec: ", slide_rec.shape)
    # --- None ---
    if crop == None: crop = 20
    # --- define scale ---
    x = np.linspace(-(slide_rec.shape[0]/2),(slide_rec.shape[0]/2),slide_rec.shape[0]+1)[:-1][crop:-crop]
    x_raW = np.linspace(-(slide_rec.shape[0]/2),(slide_rec.shape[0]/2),slide_rec.shape[0]+1)[:-1]
    print("x_ax: ", x)
    print("x_ax: ", x.shape)
    # --- make meshgrid ---
    xv, yv = np.meshgrid(x_raW, x_raW)
    yv = np.flip(yv,axis=0)
    print("xv: ", xv.shape)
    print("xv: ", xv[0:20,0])
    print("yv: ", yv[0:20,0])
    # --- calculate polynomial ---
    slide_vertical = slide_rec[:,int(slide_rec.shape[1]/2)-10:int(slide_rec.shape[1]/2)+10]
    slide_horizontal = np.rot90(slide_rec[int(slide_rec.shape[0]/2)-10:int(slide_rec.shape[0]/2)+10,:])
    print("slide_vertical: ", slide_vertical.shape)
    print("slide_horizontal: ", slide_horizontal.shape)
    slide_mean = np.mean(np.concatenate((slide_vertical,slide_horizontal),axis=1),axis=1)[crop:-crop]
    #slide_gauss = gaussian_filter(slide_mean, sigma=1000)
    slide_gauss = median_filter(slide_mean, size=200)
    print("slide_mean: ", slide_mean.shape)
    #plt.plot(x, slide_mean)
    #plt.plot(x, slide_gauss)
    #plt.show()
    # --- poly fit ---
    z = np.polyfit(x, slide_gauss, 13)
    print(z)
    #
    p = np.poly1d(z)
    print("p(0.5): ",p(0.5))
    #
    #plt.plot(x, slide_gauss)
    #plt.plot(x, p(x[:]))
    #plt.plot(x, slide_gauss-p(x[:]))
    #plt.plot(x, slide_mean-p(x[:]))
    #plt.show()
    #'''
    # ---------------------
    array_poly_bh = np.zeros_like(slide_rec)
    array_slides_bh = np.zeros_like(slide_rec_3D)
    # --- bh correction array ---
    for yy in range(slide_rec.shape[0]):
        if yy%500 == 0:
            print("yy: ", yy, " / ", slide_rec.shape[0])
        for xx in range(slide_rec.shape[1]):
            dist = np.sqrt(xv[yy,xx]**2 + yv[yy,xx]**2)
            #dist = np.sqrt(xv[xx,yy]**2 + yv[xx,yy]**2)
            #print(dist)
            #array_poly_bh[yy,xx] = slide_rec[yy,xx]-p(dist)+1
            array_poly_bh[yy,xx] = p(dist)
    array_poly_bh = array_poly_bh - np.min(array_poly_bh[int(array_poly_bh.shape[0]/2)-200:int(array_poly_bh.shape[0]/2)+200,int(array_poly_bh.shape[1]/2)-200:int(array_poly_bh.shape[1]/2)+200])
    #plt.plot(array_poly_bh[int(array_poly_bh.shape[1]/2),crop:-crop])
    #plt.show()
    # --- correct bh all slides ---
    for i_slide in range(slide_rec_3D.shape[0]):
        if i_slide%50 == 0:
            print("i_slide: ", i_slide)
        array_slides_bh[i_slide,:,:] = slide_rec_3D[i_slide,:,:] - array_poly_bh*1.0
    return array_slides_bh, array_poly_bh, xv, yv
    #'''


def beam_hardening_with_poly_array(slide_rec_3D, array_poly_bh):
    array_slides_bh = np.zeros_like(slide_rec_3D)
    # --- correct bh all slides ---
    for i_slide in range(slide_rec_3D.shape[0]):
        if i_slide%50 == 0:
            print("i_slide: ", i_slide)
        array_slides_bh[i_slide,:,:] = slide_rec_3D[i_slide,:,:] - array_poly_bh*1.0
    return array_slides_bh

def reconstruction_iteration(idx,sinograms,center,angles):
    rec_img = rec.dfi_reconstruction(sinograms[idx,:,:],center,angles=angles)
    #print(f"Processing iteration {idx} on process {os.getpid()}")
    return rec_img


def tomo_clip(data,in_min, in_max):
    data_clipped = np.clip(data,in_min,in_max)
    data_normalized = (data_clipped - in_min)/(in_max - in_min)
    data_16bit = (data_normalized*65535).astype(np.uint16)
    return data_16bit


In [9]:
import os
import time
import multiprocessing as mp

import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm

import algotom.io.loadersaver as losa
import algotom.prep.calculation as calc
import algotom.prep.removal as remo
import algotom.rec.reconstruction as rec
import algotom.util.utility as util

# import your own helpers as before
# import neutron_cleaning as nc

N_CORE = 48
APPLY_LOG = True         
FILTER_NAME = "hann"


# ----------------------------------------------------------------------
# 1) Tilt + centre of rotation
# ----------------------------------------------------------------------
def find_tilt_and_center(proj, num_rows=12, margin=100,
                         search_half_width=300, full_360=True, ncore=N_CORE):
    
    n_ang, n_rows, n_cols = proj.shape
    half = n_ang // 2 if full_360 else n_ang
    mid_col = n_cols // 2
    rows = np.linspace(margin, n_rows - 1 - margin, num_rows).astype(int)

    def _center_one_row(r):
        return calc.find_center_vo(proj[:half, r, :],
                                   start=mid_col - search_half_width,
                                   stop=mid_col + search_half_width)

    centers = Parallel(n_jobs=min(ncore, num_rows), prefer="threads")(
        delayed(_center_one_row)(r) for r in rows)
    centers = np.asarray(centers, dtype=float)

    # robust fit: drop rows whose centre is far from the median
    # (rows without sample material give garbage centres)
    med = np.median(centers)
    good = np.abs(centers - med) < 20          # px tolerance
    if good.sum() < 3:
        raise RuntimeError(
            f"Centre estimates are inconsistent: {np.round(centers, 1)}. "
            "Check search range / scan geometry (half-acquisition?).")

    slope, intercept = np.polyfit(rows[good], centers[good], 1)
    tilt_deg = np.degrees(np.arctan(slope))
    center_mid = slope * (n_rows / 2.0) + intercept

    print("Rows sampled          :", rows.tolist())
    print("Centres found         :", np.round(centers, 2).tolist())
    print("Rows used in fit      :", rows[good].tolist())
    print(f"Fitted tilt           : {tilt_deg:.4f} deg")
    print(f"Centre @ middle row   : {center_mid:.2f}")
    return tilt_deg, center_mid, rows, centers


def check_half_acquisition(proj, row=None):
    """
    For 0-360 scans: check whether this is a half-acquisition (offset-CoR)
    scan.  Returns (cor, overlap, side, overlap_metric).  If `side` puts
    the axis far from the detector centre, the sinograms must be converted
    360 -> 180 before reconstruction (see conv.convert_sinogram_360_to_180).
    """
    if row is None:
        row = proj.shape[1] // 2
    result = calc.find_center_360(proj[:, row, :], win_width=100)
    print("find_center_360 (cor, overlap, side, metric):", result)
    return result


def center_visual_scan(sinogram, out_folder, center_guess, half_range=20,
                       step=1.0, angles=None):
    """
    Brute-force sanity check: reconstruct ONE slice with a series of
    centres and save them as TIFFs -- pick the best by eye.
    `sinogram` shape: (n_angles, n_cols), full angular range.
    """
    os.makedirs(out_folder, exist_ok=True)
    centers = np.arange(center_guess - half_range,
                        center_guess + half_range + step, step)
    for c in tqdm(centers, desc="centre scan"):
        img = rec.fbp_reconstruction(sinogram, c, angles=angles,
                                     filter_name=FILTER_NAME,
                                     apply_log=APPLY_LOG, gpu=False)
        losa.save_image(os.path.join(out_folder,
                                     f"center_{c:07.2f}.tif"), img)
    print(f"Saved {len(centers)} test slices to {out_folder}")


def correct_tilt(proj, tilt_deg, ncore=N_CORE, order=1, threshold=0.02):
    """
    Rotate every projection image to make the rotation axis vertical.
    Skipped if |tilt| < threshold (deg).  NOTE on sign: if the residual
    tilt printed afterwards in main() gets larger instead of smaller,
    flip the sign here (use +tilt_deg).
    """
    if abs(tilt_deg) < threshold:
        print(f"Tilt {tilt_deg:.4f} deg below threshold -> no correction")
        return proj

    def _rot(i):
        return ndi.rotate(proj[i], -tilt_deg, reshape=False,
                          order=order, mode="nearest")

    print(f"Correcting tilt of {tilt_deg:.4f} deg on "
          f"{proj.shape[0]} projections ...")
    out = Parallel(n_jobs=ncore, prefer="threads")(
        delayed(_rot)(i) for i in range(proj.shape[0]))
    return np.asarray(out, dtype=proj.dtype)


# ----------------------------------------------------------------------
# 2) Reconstruction back-ends
# ----------------------------------------------------------------------
def _gpu_backend():
    """Pick a GPU backend if one is available, else None."""
    try:
        import astra                       # noqa: F401
        return "astra"
    except ImportError:
        pass
    try:
        from numba import cuda
        if cuda.is_available():
            return "algotom-cuda"
    except Exception:
        pass
    return None


def reconstruct_gpu(sinograms, center, angles, backend):
    """sinograms: (n_slices, n_angles, n_cols). Slice loop on the GPU."""
    n_slices, _, width = sinograms.shape
    reco = np.zeros((n_slices, width, width), dtype="f4")
    for i in tqdm(range(n_slices), desc=f"GPU reco ({backend})"):
        if backend == "astra":
            reco[i] = rec.astra_reconstruction(
                sinograms[i], center, angles=angles, method="FBP_CUDA",
                filter_name=FILTER_NAME, apply_log=APPLY_LOG)
        else:
            reco[i] = rec.fbp_reconstruction(
                sinograms[i], center, angles=angles, gpu=True,
                filter_name=FILTER_NAME, apply_log=APPLY_LOG)
    return reco


def reconstruct_cpu_gridrec(proj, center, angles, ncore=N_CORE):
    """Fastest CPU path: tomopy's gridrec, natively multi-threaded.
    proj: (n_angles, n_rows, n_cols) -- NO transpose needed."""
    import tomopy
    data = proj
    if APPLY_LOG:
        data = tomopy.minus_log(proj, ncore=ncore)
    reco = tomopy.recon(data, theta=angles, center=center,
                        algorithm="gridrec", filter_name="parzen",
                        ncore=ncore)
    return reco.astype("f4")


def _recon_chunk(sinograms, indices, center, angles):
    width = sinograms.shape[2]
    out = np.empty((len(indices), width, width), dtype="f4")
    for j, i in enumerate(indices):
        out[j] = rec.fbp_reconstruction(
            sinograms[i], center, angles=angles, gpu=False, ncore=1,
            filter_name=FILTER_NAME, apply_log=APPLY_LOG)
    return out


def reconstruct_cpu_joblib(sinograms, center, angles, ncore=N_CORE):
    """Chunked multiprocessing FBP. joblib memory-maps `sinograms` once,
    so the big array is NOT pickled per task (the main flaw of the old
    partial()-based pool)."""
    n_slices, _, width = sinograms.shape
    chunks = np.array_split(np.arange(n_slices), ncore * 2)
    chunks = [c for c in chunks if len(c)]
    results = Parallel(n_jobs=ncore, max_nbytes="1M", verbose=5)(
        delayed(_recon_chunk)(sinograms, c, center, angles) for c in chunks)
    reco = np.empty((n_slices, width, width), dtype="f4")
    for c, r in zip(chunks, results):
        reco[c] = r
    return reco


# ----------------------------------------------------------------------
# 3) Parallel saving
# ----------------------------------------------------------------------
def save_stack(folder, stack, ncore=16):
    os.makedirs(folder, exist_ok=True)
    Parallel(n_jobs=ncore, prefer="threads")(
        delayed(losa.save_image)(os.path.join(folder, f"Slice_{i:04d}.tif"),
                                 stack[i])
        for i in range(stack.shape[0]))


# ----------------------------------------------------------------------
# Main pipeline
# ----------------------------------------------------------------------
def algotom_rec(neutron_folder, name="output", use_gpu=True):
    t0 = time.time()
    sino = load_image_stack(neutron_folder)                    # noqa: F821
    pixel_size = 20.8
    print("pixel size:", pixel_size, "| stack shape:", sino.shape)
    print(f"Available cores: {mp.cpu_count()}, using: {N_CORE}")

    # -- 01 denoising --------------------------------------------------
    print("start denoising raw")
    sino_denoised = nc.remove_outliers_stack_multi(                # noqa: F821
        stack=sino, radius=2, threshold=50, core_numb=N_CORE)
    del sino
    print("end denoising raw")

    # -- 02 normalisation (currently disabled, as in original) --------
    sino_norm = sino_denoised
    del sino_denoised

    # -- 03 ring filtering ---------------------------------------------
    print("start ring filtering")
    proj_ring = util.parallel_process_slices(
        sino_norm, remo.remove_stripe_based_normalization, [15],
        ncore=N_CORE, prefer="threads")
    del sino_norm
    print("end ring filtering")

    # -- 04 tilt + centre of rotation ----------------------------------
    # If unsure about the scan geometry, run once:
    #   check_half_acquisition(proj_ring)
    tilt_deg, center, _, _ = find_tilt_and_center(proj_ring, full_360=True)
    proj_ring = correct_tilt(proj_ring, tilt_deg)

    # Optional visual verification of the centre on one slice:
    #   row = proj_ring.shape[1] // 2
    #   n_ang = proj_ring.shape[0]
    #   ang = np.linspace(0, 2*np.pi, n_ang, endpoint=False)
    #   center_visual_scan(proj_ring[:, row, :], f"{name}_center_scan",
    #                      center, half_range=25, step=1.0, angles=ang)

    if abs(tilt_deg) >= 0.02:
        # verify: residual tilt after correction should be ~0
        res_tilt, center, _, _ = find_tilt_and_center(proj_ring, num_rows=5)
        print(f"Residual tilt after correction: {res_tilt:.4f} deg "
              "(if this grew, flip the sign in correct_tilt)")

    # -- 05 reconstruction ----------------------------------------------
    n_angles = proj_ring.shape[0]
    # endpoint=False assumes projection 0 deg == 360 deg not both recorded;
    # keep endpoint=True if your scan really includes both end angles.
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)

    backend = _gpu_backend() if use_gpu else None
    t_rec = time.time()
    if backend is not None:
        print(f"Reconstructing on GPU via {backend}")
        sinograms = np.ascontiguousarray(np.transpose(proj_ring, (1, 0, 2)))
        reco = reconstruct_gpu(sinograms, center, angles, backend)
        del sinograms
    else:
        try:
            print("Reconstructing on CPU with tomopy gridrec "
                  f"({N_CORE} cores)")
            reco = reconstruct_cpu_gridrec(proj_ring, center, angles)
        except ImportError:
            print(f"tomopy not found -> joblib FBP on {N_CORE} cores")
            sinograms = np.ascontiguousarray(
                np.transpose(proj_ring, (1, 0, 2)))
            reco = reconstruct_cpu_joblib(sinograms, center, angles)
            del sinograms
    del proj_ring
    print(f"Reconstruction done in {time.time() - t_rec:.1f} s")

    # -- 06 quick QC plots ----------------------------------------------
    # plt.imshow(reco[reco.shape[0] // 2]); plt.colorbar(); plt.show(); plt.close()
    # plt.hist(reco.ravel()[::97], bins=500)   # subsampled -> much faster
    # plt.yscale("log"); plt.show(); plt.close()

    # -- 07 saving --------------------------------------------------------
    tomo16bit = tomo_clip(reco, -1.5, 2)                       # noqa: F821
    os.mkdir(str(neutron_folder)[:29]+"Reco16/"+str(neutron_folder)[29:-5])
    for img in range(reco.shape[0]):
        losa.save_image(str(neutron_folder)[:29]+"Reco16/"+str(neutron_folder)[29:-5]+"/Slice_"+str(img)+".tif",tomo16bit[img])
    print("saving finished")
    print(f"saving finished, total time {time.time() - t0:.1f} s")

In [10]:
ROOT_DIR = Path("/tmp2/Oriol/Liqiang_ILL/July")
neutron_folders = find_neutron_folders(ROOT_DIR)
#for i, folder in enumerate(neutron_folders):
#    print(f"[{i}] {folder}")
for folder in neutron_folders:    
    algotom_rec(folder)

Loaded 1200 images from /tmp2/Oriol/Liqiang_ILL/July/009_C1S3_Neutrons/Proj -> stack shape (1200, 2048, 2048), dtype uint16
pixel size: 20.8 | stack shape: (1200, 2048, 2048)
Available cores: 48, using: 48
start denoising raw
outlier filtering done!
end denoising raw
start ring filtering
end ring filtering
Rows sampled          : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Centres found         : [938.75, 939.5, 936.0, 935.0, 933.0, 930.5, 928.75, 927.75, 925.5, 923.75, 922.5, 921.75]
Rows used in fit      : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Fitted tilt           : -0.5853 deg
Centre @ middle row   : 930.22
Correcting tilt of -0.5853 deg on 1200 projections ...
Rows sampled          : [100, 561, 1023, 1485, 1947]
Centres found         : [930.5, 929.5, 929.5, 930.0, 930.75]
Rows used in fit      : [100, 561, 1023, 1485, 1947]
Fitted tilt           : 0.0124 deg
Centre @ middle row   : 930.05
Residual tilt after correction: 0.0124 de

GPU reco (astra): 100%|█████████████████████████████████████████████████████████| 2048/2048 [07:12<00:00,  4.73it/s]


Reconstruction done in 437.3 s
saving finished
saving finished, total time 787.1 s
Loaded 1200 images from /tmp2/Oriol/Liqiang_ILL/July/011_C1S3_Neutrons/Proj -> stack shape (1200, 2048, 2048), dtype uint16
pixel size: 20.8 | stack shape: (1200, 2048, 2048)
Available cores: 48, using: 48
start denoising raw
outlier filtering done!
end denoising raw
start ring filtering
end ring filtering
Rows sampled          : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Centres found         : [941.25, 940.75, 936.75, 934.75, 932.25, 930.25, 928.75, 927.25, 927.0, 924.0, 922.5, 921.25]
Rows used in fit      : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Fitted tilt           : -0.6306 deg
Centre @ middle row   : 930.55
Correcting tilt of -0.6306 deg on 1200 projections ...
Rows sampled          : [100, 561, 1023, 1485, 1947]
Centres found         : [929.75, 929.0, 929.75, 930.5, 931.25]
Rows used in fit      : [100, 561, 1023, 1485, 1947]
Fitted tilt       

GPU reco (astra): 100%|█████████████████████████████████████████████████████████| 2048/2048 [07:35<00:00,  4.50it/s]


Reconstruction done in 459.7 s
saving finished
saving finished, total time 852.9 s
Loaded 1200 images from /tmp2/Oriol/Liqiang_ILL/July/013_C1S3_Neutrons/Proj -> stack shape (1200, 2048, 2048), dtype uint16
pixel size: 20.8 | stack shape: (1200, 2048, 2048)
Available cores: 48, using: 48
start denoising raw
outlier filtering done!
end denoising raw
start ring filtering
end ring filtering
Rows sampled          : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Centres found         : [937.75, 939.75, 935.25, 935.0, 931.75, 930.25, 929.25, 927.0, 925.75, 923.5, 922.75, 921.0]
Rows used in fit      : [100, 267, 435, 603, 771, 939, 1107, 1275, 1443, 1611, 1779, 1947]
Fitted tilt           : -0.5737 deg
Centre @ middle row   : 929.91
Correcting tilt of -0.5737 deg on 1200 projections ...
Rows sampled          : [100, 561, 1023, 1485, 1947]
Centres found         : [929.25, 929.25, 930.0, 930.25, 930.75]
Rows used in fit      : [100, 561, 1023, 1485, 1947]
Fitted tilt       

...

GPU reco (astra): 100%|█████████████████████████████████████████████████████████| 2048/2048 [07:29<00:00,  4.56it/s]


Reconstruction done in 453.7 s
saving finished
saving finished, total time 847.9 s
Loaded 1200 images from /tmp2/Oriol/Liqiang_ILL/July/017_C1S3_Neutrons/Proj -> stack shape (1200, 2048, 2048), dtype uint16
pixel size: 20.8 | stack shape: (1200, 2048, 2048)
Available cores: 48, using: 48
start denoising raw


Process ForkPoolWorker-203:
Traceback (most recent call last):
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
KeyboardInterrupt
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/queues.py", line 385, in get
    res = self._reader.recv_bytes()
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/connection.py", line 437, in _recv_bytes
    return self._recv(size)
           ~~~~~~~~~~^^^^^^
  File "/home/a37

KeyboardInterrupt: 

  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ~~~~^^^^^^^^^^^^^^^
  File "/tmp2/Oriol/Urama_algotom/recast_v2.py", line 172, in remove_outliers
    median_filtered = ndi.median_filter(image, footprint=footprint)
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/site-packages/scipy/ndimage/_filters.py", line 1684, in median_filter
    return _rank_filter(input, 0, size, footprint, output, mode, cval,
                        origin, 'median', axes=axes)
  File "/home/a3723/anaconda3/envs/Tomo/lib/python3.13/site-packages/scipy/ndimage/_filters.py", line 1580, in _rank_filter
    _nd_image.rank_filter(input, rank, footprint, output, mode, cval, origins)
    ~~~~~~~~~~~~~~~~~~~~~^^